# 04 Infer Cluster ADS
Bestes Modell laden, ADS blockweise scoren, DBSCAN-Clustering, Exporte.

In [1]:
from pathlib import Path
import sys

RUN_STAGE = "smoke"  # smoke|mini|mid|full
PATHS_CONFIG = "configs/paths.local.yaml"  # alternativ: configs/paths.colab.yaml

def _find_root(start: Path) -> Path:
    for c in [start, *start.parents]:
        if (c / "src").exists() and (c / "configs").exists():
            return c.resolve()
    return start.resolve()

ROOT = _find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("ROOT:", ROOT)

RUN_ID = None  # Optional override. Default: auto from notebook 00 latest_run.json
DEVICE = "auto"
import json
import numpy as np
import pandas as pd


ROOT: C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation


In [2]:
from src.common.config import (
    load_notebook_context,
    build_run_dirs,
    resolve_run_id,
    write_run_consistency,
)
from src.common.io_schema import read_parquet
from src.approaches.nand.infer_pairs import score_pairs_with_checkpoint
from src.approaches.nand.cluster import cluster_blockwise_dbscan, resolve_dbscan_eps
from src.approaches.nand.export import build_publication_author_mapping

CTX = load_notebook_context(run_stage=RUN_STAGE, project_root=ROOT, paths_config=PATHS_CONFIG)
DATA = CTX["DATA"]
ART = CTX["ART"]
CLUSTER_CFG = CTX["CLUSTER"]

latest_context_path = Path(ART["metrics_dir"]) / "latest_run.json"
RUN_ID = resolve_run_id(RUN_ID, latest_context_path, allow_placeholder=False)
RUN_DIRS = build_run_dirs(DATA, ART, RUN_ID)
for key in ["subset_cache", "embeddings", "checkpoints", "pair_scores", "clusters", "metrics"]:
    RUN_DIRS[key].mkdir(parents=True, exist_ok=True)

write_run_consistency(
    run_id=RUN_ID,
    run_stage=RUN_STAGE,
    run_dirs=RUN_DIRS,
    output_path=RUN_DIRS["metrics"] / "04_run_consistency.json",
    extras={"notebook": "04_infer_cluster_ads", "latest_context_path": str(latest_context_path)},
)

subset_dir = RUN_DIRS["subset_cache"]
emb_dir = RUN_DIRS["embeddings"]
pair_score_dir = RUN_DIRS["pair_scores"]
cluster_dir = RUN_DIRS["clusters"]
metrics_dir = RUN_DIRS["metrics"]

ads_mentions = read_parquet(subset_dir / f"ads_mentions_{RUN_STAGE}.parquet")
ads_pairs = read_parquet(subset_dir / f"ads_pairs_{RUN_STAGE}.parquet")
ads_chars = np.load(emb_dir / f"ads_chars2vec_{RUN_STAGE}.npy")
ads_text = np.load(emb_dir / f"ads_specter_{RUN_STAGE}.npy")

with (metrics_dir / "03_train_manifest.json").open("r", encoding="utf-8") as f:
    train_manifest = json.load(f)

best_checkpoint = train_manifest["best_checkpoint"]
best_threshold = float(train_manifest["best_threshold"])

cluster_cfg_used = json.loads(json.dumps(CLUSTER_CFG))
resolved_eps, eps_meta = resolve_dbscan_eps(cluster_cfg_used, cosine_threshold=best_threshold)
cluster_cfg_used["eps"] = resolved_eps

clustering_config_used_path = metrics_dir / "04_clustering_config_used.json"
with clustering_config_used_path.open("w", encoding="utf-8") as f:
    json.dump(
        {
            "run_id": RUN_ID,
            "run_stage": RUN_STAGE,
            "best_threshold": best_threshold,
            "eps_resolution": eps_meta,
            "cluster_config_used": cluster_cfg_used,
        },
        f,
        indent=2,
    )

print("RUN_ID:", RUN_ID)
print("Best checkpoint:", best_checkpoint)
print("Best threshold:", best_threshold)
print("Resolved DBSCAN eps:", resolved_eps)
print("Clustering config used:", clustering_config_used_path)


RUN_ID: smoke_20260212T200818Z_6b207060
Best checkpoint: C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation\artifacts\checkpoints\smoke_20260212T200818Z_6b207060\smoke_20260212T200818Z_6b207060_seed1.pt
Best threshold: -0.22699999999999998
Resolved DBSCAN eps: 0.85
Clustering config used: C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation\artifacts\metrics\smoke_20260212T200818Z_6b207060\04_clustering_config_used.json


In [3]:
pair_scores_path = pair_score_dir / f"ads_pair_scores_{RUN_STAGE}.parquet"
pair_scores = score_pairs_with_checkpoint(
    mentions=ads_mentions,
    pairs=ads_pairs,
    chars2vec=ads_chars,
    text_emb=ads_text,
    checkpoint_path=best_checkpoint,
    output_path=pair_scores_path,
    device=DEVICE,
)
print("Pair scores:", len(pair_scores), "->", pair_scores_path)


Pair scores: 500 -> C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation\artifacts\pair_scores\smoke_20260212T200818Z_6b207060\ads_pair_scores_smoke.parquet


C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation\src\approaches\nand\infer_pairs.py:30: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental fea

In [4]:
clusters_path = cluster_dir / f"ads_clusters_{RUN_STAGE}.parquet"
clusters = cluster_blockwise_dbscan(
    mentions=ads_mentions,
    pair_scores=pair_scores,
    cluster_config=cluster_cfg_used,
    output_path=clusters_path,
)
print("Clusters:", len(clusters), "->", clusters_path)


Clusters: 1000 -> C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation\artifacts\clusters\smoke_20260212T200818Z_6b207060\ads_clusters_smoke.parquet


In [5]:
merged = ads_mentions[["mention_id", "block_key"]].merge(clusters, on=["mention_id", "block_key"], how="left")
cluster_size = merged.groupby(["block_key", "author_uid"]).size().rename("size").reset_index()

singleton_ratio = float((cluster_size["size"] == 1).mean()) if len(cluster_size) else 0.0
print("Singleton ratio:", singleton_ratio)
print("Cluster size describe:")
print(cluster_size["size"].describe())
display(cluster_size.groupby("block_key")["size"].agg(["count", "max"]).sort_values(["count", "max"], ascending=False).head(20))

diag = pair_scores.merge(
    clusters[["mention_id", "author_uid"]].rename(columns={"mention_id": "mention_id_1", "author_uid": "author_uid_1"}),
    on="mention_id_1",
    how="left",
).merge(
    clusters[["mention_id", "author_uid"]].rename(columns={"mention_id": "mention_id_2", "author_uid": "author_uid_2"}),
    on="mention_id_2",
    how="left",
)
diag["same_cluster"] = diag["author_uid_1"] == diag["author_uid_2"]

merged_low_conf = diag[(diag["same_cluster"]) & (diag["cosine_sim"] < best_threshold)].sort_values("cosine_sim").head(20)
split_high_sim = diag[(~diag["same_cluster"]) & (diag["cosine_sim"] >= best_threshold)].sort_values("cosine_sim", ascending=False).head(20)

print("Merged despite low confidence (top 20):")
display(merged_low_conf[["pair_id", "block_key", "cosine_sim", "distance", "author_uid_1", "author_uid_2"]])
print("Split despite high similarity (top 20):")
display(split_high_sim[["pair_id", "block_key", "cosine_sim", "distance", "author_uid_1", "author_uid_2"]])

with (metrics_dir / "04_cluster_qc.json").open("w", encoding="utf-8") as f:
    json.dump(
        {
            "singleton_ratio": singleton_ratio,
            "merged_low_conf_count": int(len(merged_low_conf)),
            "split_high_sim_count": int(len(split_high_sim)),
            "cluster_count": int(len(cluster_size)),
        },
        f,
        indent=2,
    )


Singleton ratio: 0.4423676012461059
Cluster size describe:
count    642.000000
mean       1.557632
std        0.497055
min        1.000000
25%        1.000000
50%        2.000000
75%        2.000000
max        2.000000
Name: size, dtype: float64


,count,max
block_key,,
a.clark,2,1
a.dalgarno,2,1
a.davis,2,1
a.jones,2,1
a.linde,2,1
a.smith,2,1
a.thompson,2,1
a.wilson,2,1
a.wolfendale,2,1


Merged despite low confidence (top 20):


,pair_id,block_key,cosine_sim,distance,author_uid_1,author_uid_2


Split despite high similarity (top 20):


,pair_id,block_key,cosine_sim,distance,author_uid_1,author_uid_2
459,1992PhLB..295..357A::134__1993PhRvB..47.6250I::3,t.kobayashi,0.148203,0.851797,t.kobayashi::0,t.kobayashi::1
357,1991JSG....13..539M::1__2000AIPC..523..473W::0,p.williams,0.142217,0.857783,p.williams::0,p.williams::1
107,1977Natur.266..241S::5__1982ARA&A..20....1H::0,f.hoyle,0.140832,0.859168,f.hoyle::0,f.hoyle::1
28,1974JPhA....7..120S::2__1990A&A...236..237I::2,a.wolfendale,0.139040,0.860960,a.wolfendale::0,a.wolfendale::1
379,1973Natur.241...95H::0__1999PhRvL..82.2808A::163,r.harris,0.138729,0.861271,r.harris::0,r.harris::1
234,1976NuPhB.116..185F::1__1988ApJ...332..762D::1,j.taylor,0.135467,0.864533,j.taylor::0,j.taylor::1
56,1996ApJ...470L.123P::1__1997JGR...10220439V::1,c.wilson,0.132439,0.867561,c.wilson::0,c.wilson::1
4,1969RSPSA.313..249D::0__1995JGR...10010057S::2,a.davis,0.130534,0.869466,a.davis::0,a.davis::1
90,1969JChPh..50.4074K::1__1993E&PSL.114..315A::1,d.walker,0.125916,0.874084,d.walker::0,d.walker::1
450,1973ApJ...183...49W::0__1975PhRvD..11.3583W::0,s.weinberg,0.124064,0.875936,s.weinberg::0,s.weinberg::1


In [6]:
mention_export = cluster_dir / f"mention_author_uid_{RUN_STAGE}.parquet"
pub_export = cluster_dir / f"publication_authors_{RUN_STAGE}.parquet"

clusters.to_parquet(mention_export, index=False)
pub_map = build_publication_author_mapping(mentions=ads_mentions, clusters=clusters, output_path=pub_export)

print("mention export:", mention_export)
print("publication export:", pub_export)
print("publication rows:", len(pub_map))


mention export: C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation\artifacts\clusters\smoke_20260212T200818Z_6b207060\mention_author_uid_smoke.parquet
publication export: C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation\artifacts\clusters\smoke_20260212T200818Z_6b207060\publication_authors_smoke.parquet
publication rows: 1000
